In [ ]:
rm(list=ls())

In [ ]:
library(tidyverse)
library(survival)
library(mlr3verse)
library(mlr3verse)
library(mlr3extralearners)
library(readxl)
setwd("/home/cai/R-projects")
source("tidyfuncs4sa.R")
model_name <- "deephit"
set.seed(1)

In [ ]:
data_drop <- read_excel('drop.xlsx')

In [ ]:
traindata_<-read.csv("traindataDF.csv",sep=',',header=TRUE,encoding = "UTF-8")
testdata_<-read.csv("testdataDF.csv",sep=',',header=TRUE,encoding = "UTF-8")
multidata_<-read.csv("multidataDF.csv",sep=',',header=TRUE,encoding = "UTF-8")
nacdata_<-read.csv("nacdataDF.csv",sep=',',header=TRUE,encoding = "UTF-8")

traindata_ <- traindata_[!(traindata_$ID %in% data_drop$ID), ]
testdata_ <- testdata_[!(testdata_$ID %in% data_drop$ID), ]

rownames(traindata_) <- traindata_[, 1]
traindata <- traindata_[, -1]
rownames(testdata_) <- testdata_[, 1]
testdata <- testdata_[, -1]
rownames(multidata_) <- multidata_[, 1]
multidata <- multidata_[, -1]
rownames(nacdata_) <- nacdata_[, 1]
nacdata <- nacdata_[, -1]

itps <- c(12 * c(1,2,3,4,5))

In [ ]:
library(recipes)
datarecipe_deephit <- recipe(time + status ~ ., traindata) %>%
  step_dummy(all_nominal_predictors()) %>%
  step_normalize(all_predictors()) %>%
  prep()

In [ ]:
traindata2 <- bake(datarecipe_deephit, new_data = NULL) %>%
  dplyr::select(time, status, everything())
testdata2 <- bake(datarecipe_deephit, new_data = testdata) %>%
  dplyr::select(time, status, everything())
multidata2 <- bake(datarecipe_deephit, new_data = multidata) %>%
  dplyr::select(time, status, everything())
nacdata2 <- bake(datarecipe_deephit, new_data = nacdata) %>%
  dplyr::select(time, status, everything())

In [ ]:
library(survivalmodels)
library(mlr3)
library(mlr3proba)

In [ ]:

task_train <- as_task_surv(
  traindata2, 
  time = "time",
  event = "status", 
  type = "right"
)

In [ ]:
task_test <- as_task_surv(
  testdata2, 
  time = "time",
  event = "status", 
  type = "right"
)

In [ ]:
task_multi <- as_task_surv(
  multidata2, 
  time = "time",
  event = "status", 
  type = "right"
)

In [ ]:
task_nac <- as_task_surv(
  nacdata2, 
  time = "time",
  event = "status", 
  type = "right"
)

In [ ]:

learner_deephit_0 <- lrn(
  paste0("surv.",model_name),
  optimizer = "adam",
  frac = 0.2,
  early_stopping = TRUE,
  patience = 60,
  #batch_size = 10,
  epochs = 80
)

In [ ]:

library(paradox)
search_space_deephit <- ps(
  num = p_int(lower = 1, upper = 3),
  nodes = p_int(lower = 5, upper = 13),
  learning_rate = p_dbl(lower = 0, upper = 0.2),
  dropout = p_dbl(lower = 0, upper = 0.2),
  weight_decay = p_dbl(lower = 0, upper = 0.2)
)
search_space_deephit$extra_trafo <- function(x, param_set) {
  x$num_nodes = rep(x$nodes, x$num)
  x$nodes = x$num = NULL
  return(x)
}

In [ ]:

learner_deephit <- auto_tuner(
  tuner = tnr("random_search"),
  search_space = search_space_deephit,
  learner = learner_deephit_0,
  resampling = rsmp("cv", folds = 5),
  measure = msr("surv.cindex"),
  terminator = trm("evals", n_evals = 50)
)

In [ ]:
search_space_deephit

In [ ]:
library(reticulate)
use_python('/usr/bin/python3')
source_python('setseed.py')

In [ ]:
learner_deephit$train(task_train)

In [ ]:

#learner_deephit
paste(learner_deephit$tuning_result)
# learner_deephit$tuning_instance
# autoplot(learner_deephit$tuning_instance)
#learner_deephit$tuning_instance$archive
# learner_deephit$tuning_instance$result_learner_param_vals
#learner_deephit$learner$model

In [ ]:

predtrain_deephit <- learner_deephit$predict(task_train)

In [ ]:

predtrain_deephit$score(msrs(c("surv.cindex")))

In [ ]:

predrisktrain_deephit <- 
  predtrain_deephit$crank %>%
  as.data.frame()

predprobtrain_deephit <- 
  predtrain_deephit$distr[
    1:nrow(traindata2)
  ]$survival(itps) %>%
  t() %>%
  as.data.frame() %>%
  mutate(risk=predrisktrain_deephit$.,
         dataset = "train",
         time = traindata2$time,
         status = traindata2$status)

In [ ]:
predprobtrain_deephit

In [ ]:

evaltrain_deephit <- eval4sa(
  predprob = predprobtrain_deephit,
  preddata = traindata2,
  etime = "time",
  estatus = "status",
  model = model_name,
  dataset = "train",
  timepoints = c(12,24,36,48,60),
  plotcalimethod = "quantile",
  bw4nne = NULL,
  q4quantile = 1
)
evaltrain_deephit$auc
evaltrain_deephit$roc
evaltrain_deephit$rocplot
evaltrain_deephit$brierscore
evaltrain_deephit$brierscoretest
evaltrain_deephit$calibration
evaltrain_deephit$calibrationplot

In [ ]:

predtest_deephit <- learner_deephit$predict(task_test)
testcindex <- predtest_deephit$score(msrs(c("surv.cindex")))
testcindex

In [ ]:

predrisktest_deephit <- 
  predtest_deephit$crank %>%
  as.data.frame()

predprobtest_deephit <- 
  predtest_deephit$distr[
    1:nrow(testdata2)
  ]$survival(itps) %>%
  t() %>%
  as.data.frame() %>%
  mutate(risk = predrisktest_deephit$.,
         dataset = "test",
         time = testdata2$time,
         status = testdata2$status)

In [ ]:

evaltest_deephit <- eval4sa(
  predprob = predprobtest_deephit,
  preddata = testdata2,
  etime = "time",
  estatus = "status",
  model = model_name,
  dataset = "test",
  timepoints = itps,
  plotcalimethod = "quantile",
  bw4nne = NULL,
  q4quantile = 1
)
evaltest_deephit$auc
evaltest_deephit$roc
evaltest_deephit$rocplot
evaltest_deephit$brierscore
evaltest_deephit$brierscoretest
evaltest_deephit$calibration
evaltest_deephit$calibrationplot

In [ ]:
auc_pic = evaltest_deephit$rocplot

In [ ]:
ggsave(paste0('final_result/',model_name,'_cindex_',format(testcindex, digits = 4),'.jpg'),auc_pic)

In [ ]:

predmulti_deephit <- learner_deephit$predict(task_multi)
multicindex <- predmulti_deephit$score(msrs(c("surv.cindex")))
multicindex

predriskmulti_deephit <- 
  predmulti_deephit$crank %>%
  as.data.frame()

predprobmulti_deephit <- 
  predmulti_deephit$distr[
    1:nrow(multidata2)
  ]$survival(itps) %>%
  t() %>%
  as.data.frame() %>%
  mutate(risk = predriskmulti_deephit$.,
         dataset = "External test",
         time = multidata2$time,
         status = multidata2$status)

evalmulti_deephit <- eval4sa(
  predprob = predprobmulti_deephit,
  preddata = multidata2,
  etime = "time",
  estatus = "status",
  model = model_name,
  dataset = "External test",
  timepoints = c(12,24,36,48,60),
  plotcalimethod = "quantile",
  bw4nne = NULL,
  q4quantile = 1
)
evalmulti_deephit$auc
evalmulti_deephit$roc
evalmulti_deephit$rocplot
evalmulti_deephit$brierscore
evalmulti_deephit$brierscoretest
evalmulti_deephit$calibration
evalmulti_deephit$calibrationplot

In [ ]:
prednac_deephit <- learner_deephit$predict(task_nac)
naccindex <- prednac_deephit$score(msrs(c("surv.cindex")))
naccindex


predrisknac_deephit <- 
  prednac_deephit$crank %>%
  as.data.frame()

predprobnac_deephit <- 
  prednac_deephit$distr[
    1:nrow(nacdata2)
  ]$survival(itps) %>%
  t() %>%
  as.data.frame() %>%
  mutate(risk = predrisknac_deephit$.,
         dataset = "External test",
         time = nacdata2$time,
         status = nacdata2$status)


evalnac_deephit <- eval4sa(
  predprob = predprobnac_deephit,
  preddata = nacdata2,
  etime = "time",
  estatus = "status",
  model = model_name,
  dataset = "External test",
  timepoints = c(12,24,36,48,60),
  plotcalimethod = "quantile",
  bw4nne = NULL,
  q4quantile = 1
)
evalnac_deephit$auc
evalnac_deephit$roc
evalnac_deephit$rocplot
evalnac_deephit$brierscore
evalnac_deephit$brierscoretest
evalnac_deephit$calibration
evalmulti_deephit$calibrationplot

In [ ]:
evalnac_deephit$rocplot

In [ ]:
a<-evaltrain_deephit$rocplot
b<-evaltest_deephit$rocplot
c<-evalmulti_deephit$rocplot
d<-evalnac_deephit$rocplot
ggsave("ROC_train_deephit$roc.pdf", plot = a, width = 5, height = 5)  
ggsave("ROC_test_deephit$roc.pdf", plot = b, width = 5, height = 5)  
ggsave("ROC_multi_deephit$roc.pdf", plot = c, width = 5, height = 5)  
ggsave("ROC_nac_deephit$roc.pdf", plot = c, width = 5, height = 5)  

In [ ]:

save(predtrain_deephit,
     predprobtrain_deephit,
     evaltrain_deephit,
     
     predtest_deephit,
     predprobtest_deephit,
     evaltest_deephit,
     
     predmulti_deephit,
     predprobmulti_deephit,
     evalmulti_deephit,
     
     prednac_deephit,
     predprobnac_deephit,
     evalnac_deephit,
     file = paste0("final_result/",model_name,"_mlsa_deephit.RData"))

In [ ]:
predprobtrain_deephit$ID=traindata_$ID
colnames<-colnames(predprobtrain_deephit)
colnames<-append(colnames[length(colnames)],colnames[1:length(colnames)-1])
ccolnames<-colnames[colnames!='model']
predprobtrain_deephit<-select(predprobtrain_deephit,colnames)
write.csv(predprobtrain_deephit, paste0('final_result/train_pred_',model_name,'.csv'),row.names=FALSE)

In [ ]:
predprobtest_deephit$ID=testdata_$ID
colnames<-colnames(predprobtest_deephit)
colnames<-append(colnames[length(colnames)],colnames[1:length(colnames)-1])
colnames<-colnames[colnames!='model']
predprobtest_deephit<-select(predprobtest_deephit,colnames)
write.csv(predprobtest_deephit, paste0('final_result/test_pred_',model_name,'.csv'),row.names=FALSE)

predprobmulti_deephit$ID=multidata_$ID
colnames<-colnames(predprobmulti_deephit)
colnames<-append(colnames[length(colnames)],colnames[1:length(colnames)-1])
colnames<-colnames[colnames!='model']
predprobmulti_deephit<-select(predprobmulti_deephit,colnames)
write.csv(predprobmulti_deephit, paste0('final_result/multi_pred_',model_name,'.csv'),row.names=FALSE)

predprobnac_deephit$ID=nacdata_$ID
colnames<-colnames(predprobnac_deephit)
colnames<-append(colnames[length(colnames)],colnames[1:length(colnames)-1])
colnames<-colnames[colnames!='model']
predprobnac_deephit<-select(predprobnac_deephit,colnames)
write.csv(predprobnac_deephit, paste0('final_result/nac_pred_',model_name,'.csv'),row.names=FALSE)


In [ ]:
auc1<-evaltrain_deephit$auc
auc1$index<-predtrain_deephit$score(msrs(c("surv.cindex")))
auc2<-evaltest_deephit$auc
auc2$index<-predtest_deephit$score(msrs(c("surv.cindex")))
auc3<-evalmulti_deephit$auc
auc3$index<-predmulti_deephit$score(msrs(c("surv.cindex")))
auc4<-evalnac_deephit$auc
auc4$index<-prednac_deephit$score(msrs(c("surv.cindex")))

auc<-cbind(auc1,auc2,auc3,auc4)
write.table(auc, file=paste0("final_result/auc_", model_name, '.csv'), sep=",", quote=T, row.names=F, col.names=T)

In [ ]:
paste0(learner_deephit$tuning_result)

In [ ]:
liststr <- paste(learner_deephit$tuning_result[[6]])
writeLines(substr(liststr, start = 6, stop = nchar(liststr) - 1), paste0("final_result/", model_name, '_params.txt'))

In [ ]:
testcindex
multicindex
naccindex

In [ ]:
colnames(traindata2)
traindatax <- traindata2[, 3:ncol(traindata2)]
colnames(traindatax)

exper_deephit <- survex::explain_survival(
  learner_deephit,
  data = traindatax,
  y = survival::Surv(
    time = traindata2$time,
    event = traindata2$status
  ),
  predict_function = risk_pred,
  predict_survival_function = surv_pred,
  predict_cumulative_hazard_function = chf_pred,
  label = "deephit",
  times = itps
)

In [ ]:

library(dplyr)
set.seed(3)
vip_deephit <- survex::model_parts(
  exper_deephit,
  type = "ratio",
  N = 682
)
plot(vip_deephit, max_vars = ncol(traindatax)+1)

In [ ]:
ggsave(filename = "vip_deephit_notime.pdf", plot = p, width = 8, height = 5)  

In [ ]:

library(scales)  

max_value <- vip_deephit$result %>%  
  filter(`_permutation_` == 0) %>%  
  select(all_of(colnames(traindatax))) %>%  
  pivot_longer(cols = everything()) %>%  
  summarise(max_value = max(value, na.rm = TRUE)) %>%  
  pull(max_value)  
p<-vip_deephit$result %>%  
  filter(`_permutation_` == 0) %>%  
  rename("Time" = "_times_") %>%  
  select(Time, all_of(colnames(traindatax))) %>%  
  pivot_longer(cols = -1) %>%  
  mutate(name = tidytext::reorder_within(name, value, Time)) %>%  
  ggplot(aes(x = value, y = name)) +  
  geom_col(fill = "#C85D4D") +  
  # geom_vline(xintercept = 1) +  
  tidytext::scale_y_reordered() +  
  labs(x = "importance", y = "") +  
  facet_wrap(~Time, scales = "free_y", labeller = "label_both") +  
  scale_x_continuous(limits = c(1, max_value + 0.02), oob = scales::squish) +   
  theme_minimal() +  
  theme(  
    text = element_text(size =7)  # Setting the font size to 3 for all text elements  
  )   
p

In [ ]:

shaps_deephit_df <- traindatax %>%
  pmap_dfr(
    function(...) {
      current <- tibble(...)
      set.seed(1)
      shapi_deephit <- survex::predict_parts(
        exper_deephit,
        new_observation = current,
        N = 682,
        calculation_method = "kernelshap"
      )
      return(shapi_deephit$result %>% rownames_to_column("Time"))
    }
  )

In [ ]:
shaps_deephit_df2 <- shaps_deephit_df %>%
  mutate(id = rep(1:nrow(traindatax), each = length(itps))) %>%
  select(id, Time, everything()) %>%
  pivot_longer(cols = -c(1,2), values_to = "shapley")

shapleyimp_deephit <- shaps_deephit_df2 %>%
  group_by(name) %>%
  summarise(shapley.abs.mean = mean(abs(shapley), na.rm = T)) %>%
  arrange(shapley.abs.mean) %>%
  mutate(name = as_factor(name))

In [ ]:
flname <- colnames(traindatax)[c(1:27)]
lxname <- colnames(traindatax)[c(1:27)]
flname

In [ ]:
library(ggh4x)
traindatax %>%
  mutate(id = 1:n()) %>%
  select(id, all_of(flname)) %>%
  pivot_longer(cols = -1) %>%
  left_join(shaps_deephit_df2, by = c("id", "name")) %>%
  mutate(Time = as_factor(Time)) %>%
  ggplot(aes(x = interaction(value, name), y = shapley)) +
  geom_boxplot(aes(fill = name), show.legend = F) +
  geom_hline(yintercept = 0, color = "grey10") +
  scale_x_discrete(NULL, guide = "axis_nested") +
  scale_colour_viridis_c() +
  labs(x = "") +
  facet_wrap(~Time, ncol = 1, scales = "free_y") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1),
        legend.position = "bottom")

In [ ]:
shapplot<-traindatax %>%
  mutate(id = 1:n()) %>%
  select(id, all_of(lxname)) %>%
  pivot_longer(cols = -1) %>%
  left_join(shaps_deephit_df2, by = c("id", "name")) %>%
  mutate(Time = as_factor(Time)) %>%
  dplyr::group_by(name) %>%
  dplyr::mutate(
    value = (value - min(value)) / (max(value) - min(value)),
    name = factor(name, levels = levels(shapleyimp_deephit$name))
  ) %>%
  dplyr::arrange(value) %>%
  dplyr::ungroup() %>%
  ggplot(aes(x = shapley, y = name, color = value)) +
  ggbeeswarm::geom_quasirandom(width = 0.2) +
  scale_color_gradient(
    low = "red",
    high = "blue",
    breaks = c(0, 1),
    labels = c("Low", "High"),
    guide = guide_colorbar(barwidth = 0.5,
                           barheight = length(lxname)*2,
                           ticks = F,
                           title.position = "right",
                           title.hjust = 0.5)
  ) +
  labs(x = "SHAP value", color = "Feature value") +
  #facet_wrap(~Time) +
  theme_bw() +
  theme(legend.title = element_text(angle = -90))

In [ ]:
ggsave(filename = "SHAP_deephit.pdf", plot = shapplot, width = 7, height = 5)  